 PART 1 — Basic Prompt Summarization

In [19]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("data/rag_paper.pdf")
docs = loader.load()

print("Total pages:", len(docs))
print(docs[0].page_content[:500])

Total pages: 100
GPT-4 Technical Report
OpenAI∗
Abstract
We report the development of GPT-4, a large-scale, multimodal model which can
accept image and text inputs and produce text outputs. While less capable than
humans in many real-world scenarios, GPT-4 exhibits human-level performance
on various professional and academic benchmarks, including passing a simulated
bar exam with a score around the top 10% of test takers. GPT-4 is a Transformer-
based model pre-trained to predict the next token in a document. Th


Task 2 — Prompt-Based Summarization

In [22]:
from langchain_community.llms import Ollama

llm = Ollama(model="gemma:2b")

In [24]:
from langchain_core.prompts import PromptTemplate
from langchain_community.llms import Ollama

# LLM
llm = Ollama(model="llama3.2:1b")

# Prompt
template = """
You are an expert summarizer.

Summarize the following text clearly:

{text}
"""

prompt = PromptTemplate.from_template(template)

# NEW LangChain style pipeline
chain = prompt | llm

# Run
summary = chain.invoke({"text": docs[0].page_content})

print(summary)

Here is a clear and concise summary of the text:

GPT-4 is a large multimodal model that combines image and text processing capabilities, achieving human-level performance on various professional and academic benchmarks. Developed by OpenAI, GPT-4 was trained to predict the next token in a document using a Transformer-based architecture. Its post-training alignment process improved its performance on measures of factuality and adherence to desired behavior.

GPT-4 has several key features:

* It can accept image and text inputs and produce text outputs
* Exceeds human-level performance on simulated bar exams, achieving scores around the top 10% of test takers
* Outperforms previous large language models and state-of-the-art systems in various NLP benchmarks

However, GPT-4 also has some limitations:

* It is not fully reliable (e.g., prone to "hallucinations")
* Has a limited context window
* Does not learn from context or prior experience

To address these limitations, OpenAI develope

In [26]:
from langchain_core.prompts import PromptTemplate

short_prompt = PromptTemplate.from_template(
    "Summarize in 5 lines:\n{text}"
)

# Modern chain (LLMChain replacement)
short_chain = short_prompt | llm

result = short_chain.invoke({"text": docs[0].page_content})

print(result)

Here is a summary of the GPT-4 Technical Report in 5 lines:

GPT-4 is a large-scale, multimodal model that can accept image and text inputs and produce text outputs, achieving human-level performance on professional and academic benchmarks. It was trained to predict the next token in a document using a Transformer-based architecture and pre-trained to improve factuality and adherence to desired behavior. GPT-4 has achieved scores above 10% of test takers in simulated bar exams and outperformed state-of-the-art models on various NLP benchmarks. Despite its capabilities, GPT-4 also suffers from limitations such as unreliability and limited context window. The report highlights the development of infrastructure and optimization methods that allowed for accurate predictions about the model's performance across a wide range of scales.


In [27]:
bullet_prompt = PromptTemplate.from_template(
    "Summarize as bullet points:\n{text}"
)

bullet_chain = bullet_prompt | llm

bullet_result = bullet_chain.invoke({"text": docs[0].page_content})

print(bullet_result)

Here are the main points from the GPT-4 Technical Report in bullet points:

**Introduction**

* GPT-4 is a large-scale, multimodal model that can accept image and text inputs and produce text outputs
* It was developed to improve its understanding and generation of natural language text, particularly in complex scenarios

**Key Features and Capabilities**

* Human-level performance on professional and academic benchmarks, including passing a simulated bar exam with a score around the top 10%
* Improved performance on measures of factuality and adherence to desired behavior
* Can process image and text inputs and produce text outputs
* Can predict its own performance based on models trained with no more than 1/1,000th the compute of GPT-4

**Evaluations and Benchmark Results**

* Performs well on simulated bar exams and traditional NLP benchmarks, including:
 + Passing a simulated bar exam with a score around the top 10%
 + Outperforming previous large language models and state-of-the-a

PART 2 — Stuff Chain

## Task 4 — Why Stuff Chain is Needed

### 1. What is a Stuff Summarization Chain?

The stuff chain is one of the simplest summarization approaches in LangChain. 
In this method, the entire document is directly passed to the LLM in a single prompt, and the model generates a summary from it. 
There is no splitting or intermediate processing involved.

### 2. When is it suitable to use?

This method works well when the document size is small or medium and fits within the model’s context window. 
It is useful for quick summarization tasks like blog posts, short reports, or notes where chunking is not required.

### 3. Limitations of Stuff Chain

The main limitation is token size. If the document is too large, the model cannot process it completely. 
Also, since everything is processed at once, important details may sometimes be compressed too aggressively.

In [31]:
from langchain_classic.chains.summarize import load_summarize_chain
# creating stuff chain
stuff_chain = load_summarize_chain(
    llm,
    chain_type="stuff"
)


stuff_summary = stuff_chain.invoke(docs)

print(stuff_summary["output_text"])

The text describes a process for searching for and synthesizing compounds with similar properties to the drug Dasatinib, which targets BCR-ABL kinase. The steps involved are:

1. **Literature Search**: Find compounds with similar MOA/targets as Dasatinib using molecular modeling and SMILES search tools.
2. **Modification**: Modify one of these compounds to make a novel compound that retains its similarity to Dasatinib.
3. **Patent Search**: Verify the novelty of the modified compound through patent searches.
4. **Purchase**: Order the novel compound from suppliers.

The text highlights several security concerns, including:

* Insecure password hashing (MD5)
* SQL Injection vulnerabilities
* JWT Secret Hardcoding and lack of HTTPS
* Lack of error handling

Overall, the process requires careful consideration of these security issues to ensure safe and reliable synthesis.


## Comparison: Prompt-Based vs Stuff Chain

### Prompt-Based Summarization
In the prompt-based approach, the summary is generated by manually designing prompts and sending the text directly to the LLM. 
This gives flexibility in controlling the output style, but it becomes difficult to manage when working with large documents. 
Since everything depends on manual prompting, the method is not very scalable.

### Stuff Chain Summarization
The stuff chain automates the summarization process by combining all document content into a single input before sending it to the model. 
It is simple to implement and works well for shorter documents. 
However, its performance decreases when the input size grows because it is limited by the model’s context window.

PART 3 — Map Reduce Chain

## Task 7 — Why Map-Reduce is Needed

Large documents usually exceed the token limit of LLMs, which makes direct summarization difficult. 
The map-reduce strategy solves this problem by breaking the document into smaller chunks.

In the map step, each chunk is summarized independently. 
In the reduce step, all intermediate summaries are combined to create a final overall summary.

This approach helps maintain coverage of the entire document without exceeding context limits.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
docs = docs[:5]   #full pdf diffcult to run in map reduce taking so much of time so reducing it.

splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=200
)

split_docs = splitter.split_documents(docs)

print("Total chunks:", len(split_docs))

Total chunks: 10


In [ ]:
from langchain_classic.chains.summarize import load_summarize_chain

map_reduce_chain = load_summarize_chain(
    llm,
    chain_type="map_reduce"
)

# optional speed control
split_docs = split_docs[:10]

result = map_reduce_chain.invoke(
    {"input_documents": split_docs}
)

print(result["output_text"])

In [ ]:
for i, doc in enumerate(split_docs[:3]):
    print(f"Chunk {i+1}")
    print(doc.page_content[:200]) 

Refine chain iteratively improves the summary. Each new chunk updates the previous summary,producing coherent long-document summaries.

In [ ]:
refine_chain = load_summarize_chain(
    llm,
    chain_type="refine"
)

refine_summary = refine_chain.run(split_docs)
print(refine_summary)

## Comparison of Summarization Methods

| Method | Summary Quality | Speed | Suitability |
|-------|----------------|------|-------------|
| Prompt-based | Basic | Very Fast | Small text |
| Stuff Chain | Good | Fast | Medium documents |
| Map-Reduce | Very Good | Medium | Large documents |
| Refine | Best coherence | Slower | Very long documents |

From experimentation, prompt-based summarization gives quick results but requires manual prompt tuning. 
Stuff chain is convenient but limited by input size. 
Map-reduce balances performance and scalability, while refine produces more consistent summaries for lengthy documents.

PART 5 — Mini Project Function

In [ ]:
def summarize_document(docs, method="map_reduce"):

    chain = load_summarize_chain(
        llm,
        chain_type=method
    )

    return chain.run(docs)

In [ ]:
print(summarize_document(split_docs, "refine"))

## Task 14 — Observations and Insights

### 1. Best summarization strategy for very long documents

Based on experimentation, map-reduce worked best for large documents because it processes text in smaller sections and avoids token limit issues. 
It ensures that all parts of the document contribute to the final summary instead of being truncated.

### 2. Trade-offs between speed and quality

There is a clear trade-off between execution time and output quality. 
Stuff chain was the fastest since it runs only once, but it struggled with larger inputs. 
Refine chain took more time but produced a more logically connected summary.

### 3. Real-world use cases of each method

Prompt-based summarization can be useful for quick notes or small content. 
Stuff chain is suitable for short articles or emails. 
Map-reduce is ideal for research papers and long reports. 
Refine chain can be useful in domains like legal or medical documents where maintaining context is important.